# Generating Psalm- and Proverb-Style Text in Spanish with a Character-Level GPT Trained from Scratch

**Name:** Arturo Ramos
**Dataset:** *Santa Biblia — Reina Valera 1909* (public domain), plain-text verse-per-line edition from eBible.org — https://ebible.org/find/details.php?id=spaRV1909 — file `data/spaRV1909_vpl.txt`

**Generative task.** This is **Transformer-based text generation**. A small decoder-only Transformer (a GPT-style language model) is trained from scratch, character by character, to generate new Spanish text in the style of the poetic and wisdom books of the 1909 Reina-Valera Bible, the Psalms (*Salmos*) and Proverbs (*Proverbios*). Training has two phases: pre-training on the whole Bible teaches the model 1909 Spanish, and fine-tuning on Psalms and Proverbs teaches it their style. Each verse is prefixed with its book code (`PSA:` or `PRO:`), so the prefix works as a prompt that asks for one style or the other. The system produces short verse-like lines; it is a study of how a generative model imitates a sacred text, not a tool for producing scripture.

## 1. Setup

In [ ]:
import os
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")   # required for deterministic cuBLAS kernels

import math
import random
import re
import time
import unicodedata
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import torch.nn.functional as F

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.width", 160)
pd.set_option("display.max_colwidth", 140)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.use_deterministic_algorithms(True, warn_only=True)
torch.backends.cudnn.benchmark = False

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DATA_FILE = Path("data") / "spaRV1909_vpl.txt"
FIG_DIR = Path("figures")
FIG_DIR.mkdir(exist_ok=True)
print("torch", torch.__version__, "| device:", DEVICE, "|", torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else "CPU")

## 2. Load and Inspect the Data

The eBible.org "VPL" edition has one verse per line in the form `BOOK chapter:verse text`, with all headings, notes and paragraph marks removed.

In [ ]:
lines = DATA_FILE.read_text(encoding="utf-8-sig").splitlines()
pattern = re.compile(r"^(\w{3}) (\d+):(\d+) (.*)$")
parsed = [pattern.match(l) for l in lines]
print("lines:", len(lines), "| lines that do not match 'BOOK c:v text':", sum(p is None for p in parsed))

verses = pd.DataFrame([p.groups() for p in parsed], columns=["book", "chapter", "verse", "text"])
verses["chapter"] = verses["chapter"].astype(int)
verses["verse"] = verses["verse"].astype(int)
verses["chars"] = verses["text"].str.len()
print("books:", verses["book"].nunique(), "| total characters:", f"{verses['chars'].sum():,}")
verses.head()

In [ ]:
# Representative samples: the two target books and a narrative book for contrast
pd.concat([verses[(verses.book == "PSA") & (verses.chapter == 23)].head(3),
           verses[(verses.book == "PRO") & (verses.chapter == 3)].iloc[4:7],
           verses[(verses.book == "GEN") & (verses.chapter == 1)].head(2)])[["book", "chapter", "verse", "text"]]

In [ ]:
target = verses[verses["book"].isin(["PSA", "PRO"])]
summary = pd.DataFrame({
    "verses": [len(verses), len(target), (verses.book == "PSA").sum(), (verses.book == "PRO").sum()],
    "characters": [verses.chars.sum(), target.chars.sum(), verses.loc[verses.book == "PSA", "chars"].sum(), verses.loc[verses.book == "PRO", "chars"].sum()],
    "median characters per verse": [verses.chars.median(), target.chars.median(), verses.loc[verses.book == "PSA", "chars"].median(), verses.loc[verses.book == "PRO", "chars"].median()],
}, index=["whole Bible", "Psalms + Proverbs", "Psalms", "Proverbs"])
summary

In [ ]:
# Character inventory and orthographic features of the 1909 text
all_text = "".join(verses["text"])
char_counts = Counter(all_text)
print("distinct characters:", len(char_counts))
print("".join(sorted(char_counts)))
features = {
    "verses with square brackets [ ]": int(verses["text"].str.contains(r"\[").sum()),
    "occurrences of the preposition 'á'": len(re.findall(r"\bá\b", all_text)),
    "occurrences of 'fué'": len(re.findall(r"\bfué\b", all_text)),
    "Psalm verses that start with a heading ('Salmo de ...')": int(verses[(verses.book == "PSA")]["text"].str.match(r"^(Salmo|Al Músico|Masquil|Mictam|Oración)").sum()),
    "verses whose first word is fully upper-case": int(verses["text"].str.match(r"^[A-ZÁÉÍÓÚÑ]{2,}\b").sum()),
}
pd.Series(features, name="count")

**Structure, formats and preprocessing considerations**

- The file has 31,102 verse lines from the 66 books, about 3.8 million characters in total and only 83 distinct characters, which makes a character-level model small and practical: the vocabulary fits in less than a hundred symbols and no word can be out of vocabulary.
- The two target books are a small part of the corpus: Psalms and Proverbs together have 3,376 verses (about 7 % of the characters). A model trained only on them would see too little Spanish to learn spelling and grammar, which motivates pre-training on the whole Bible first.
- The text keeps the 1909 orthography (for example the preposition *á* written with an accent and *fué* with an accent), which is part of the style to be imitated, so it is **not** modernized.
- Square brackets mark words added by the translators for clarity. They are editorial marks rather than language, so the brackets are removed and the words kept.
- Some verses start with a fully upper-case word (the first word of a chapter, as in *EN el principio*), and the first verse of many Psalms contains its heading (*Salmo de David.*). Both are kept: they are part of how the text looks, and the model will reproduce them.